# GPU


## 预备
首先，确保你至少安装了一个NVIDIA GPU。
然后，下载[NVIDIA驱动和CUDA](https://developer.nvidia.com/cuda-downloads)
并按照提示设置适当的路径。
当这些准备工作完成，就可以使用`nvidia-smi`命令来(**查看显卡信息。**)

In [2]:
! nvidia-smi

Sun Oct 23 18:01:42 2022       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 515.76       Driver Version: 515.76       CUDA Version: 11.7     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA GeForce ...  Off  | 00000000:01:00.0 Off |                  N/A |
|  0%   38C    P8     5W / 250W |      5MiB / 11264MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA GeForce ...  Off  | 00000000:03:00.0 Off |                  N/A |
|  0%   

在PyTorch中，每个数组都有一个设备（device），
我们通常将其称为上下文（context）。

当在带有GPU的服务器上训练神经网络时，我们通常希望模型的参数在GPU上。

在GPU上创建的张量只消耗这个GPU的显存。我们可以使用`nvidia-smi`命令查看显存使用情况。一般来说，我们需要确保不创建超过GPU显存限制的数据。

## torch.device


我们可以指定用于存储和计算的设备，如CPU和GPU。

默认情况下，所有变量和相关的计算都**分配给CPU**. `cpu`设备意味着所有物理CPU和内存，这意味着PyTorch的计算将尝试使用所有CPU核心。

然而，`gpu`设备只代表一个卡和相应的显存。如果有多个GPU，我们使用`torch.device(f'cuda:{i}')`来表示第$i$块GPU（$i$从0开始）。

另外，`cuda:0`和`cuda`是等价的。


In [9]:
import torch
from torch import nn

# 分配CPU
d1 = torch.device('cpu')

# 分配第一块GPU
d2 = torch.device('cuda')

# 分配第二块GPU
d3 = torch.device('cuda:1')

# 另一种写法
d4 = torch.device('cuda', 0)

d1, d2, d3, d4

(device(type='cpu'),
 device(type='cuda'),
 device(type='cuda', index=1),
 device(type='cuda', index=0))

### 查询可用gpu的数量


In [9]:
torch.cuda.device_count()

1

### 自适应

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

# 查询所在的设备

默认情况下，张量是在CPU上创建的。

In [6]:
import torch
from torch import nn
x = torch.tensor([1, 2, 3])
x.device

device(type='cpu')

模型

In [14]:
net = nn.Sequential(nn.Linear(3, 1))

# 没有 net.device
# 只能从模型参数看出模型在哪里
net[0].weight.data.device

device(type='cpu')

# GPU的可见性

对`to(device)`和`cuda()`都有效.

## 使用set_device

这个是全局设置

使用`torch.cuda.set_device()`表示仅某个GPU可见, 还得使用 `.cuda()`分配

In [4]:
import torch
from torch import nn

torch.cuda.set_device(1)
# torch.cuda.set_device('cuda:1')

# 张量
X = torch.ones(2, 3).cuda()
print(X, X.device)

net = nn.Sequential(nn.Linear(3, 1)).cuda()
print(net[0].weight.data.device)

net(X)

tensor([[1., 1., 1.],
        [1., 1., 1.]]) cuda:1
cuda:1


tensor([[1.1925],
        [1.1925]], grad_fn=<AddmmBackward0>)

或者

In [6]:
with torch.cuda.device(0):
   # your code to run
   ...

## 环境变量 CUDA_VISIBLE_DEVICES

> python代码中设定

In [6]:
import torch
import os 
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
# os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

print(torch.cuda.device_count())

X1 = torch.ones(2, 3).to(0)
X2 = torch.ones(2, 3).cuda()

X1.device, X2.device

2


(device(type='cuda', index=0), device(type='cuda', index=0))

> 命令行中设定
```bash
CUDA_VISIBLE_DEVICES=1 python my_script.py
```
临时的

# 分配到GPU

## PS

假设变量`Z`已经存在于第二个GPU上，并不会复制并分配新内存。


In [ ]:
K = torch.tensor([2]).cuda(0)
print(K.device)
a = K.cuda(0) is K
b = K.to(torch.device('cuda')) is K
a,b 

cuda:0


(True, True)

## 张量/模型通用

### to() 

创建张量后移动到GPU上

In [ ]:
Y = torch.rand(2, 3)

# 无参 cpu
Y2 = Y.to()

# 0
Y3 = Y.to(0)

# cuda:0
Y4 = Y.to('cuda:0')

# device
Y5 = Y.to(torch.device('cuda:0'))

Y.device, Y2.device, Y3.device, Y4.device, Y5.device

(device(type='cpu'),
 device(type='cpu'),
 device(type='cuda', index=0),
 device(type='cuda', index=0),
 device(type='cuda', index=0))

模型

In [ ]:
import torch
from torch import nn
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = nn.Sequential(nn.Linear(3, 1))

# 模型移动到GPU上
net = net.to(device)
print(net[0].weight.data.device)

# 输入的数据也要在同一块GPU上
X = torch.ones(2, 3, device=device)
print(X.device)
net(X)

cuda:0
cuda:0


tensor([[0.5285],
        [0.5285]], device='cuda:0', grad_fn=<AddmmBackward0>)

### .cuda()

In [ ]:
import torch
from torch import nn

# 张量
X = torch.ones(2, 3)
Z = X.cuda(1)
print(X.device, Z.device)

net = nn.Sequential(nn.Linear(3, 1))

# 模型移动到GPU上
net = net.cuda(1)
print(net[0].weight.data.device)

net(Z)

cpu cuda:1
cuda:1


tensor([[-0.6722],
        [-0.6722]], device='cuda:1', grad_fn=<AddmmBackward0>)

In [ ]:
X = torch.ones(2, 3)

# 分配到默认的GPU上，没有设置set_device就是默认0号GPU
X1 = X.cuda()
X2 = X.cuda(0)
X3 = X.cuda('cuda:0')

X1.device, X2.device, X3.device

(device(type='cuda', index=0),
 device(type='cuda', index=0),
 device(type='cuda', index=0))

## 特殊的操作

### 创建张量时指定存储设备
只有张量有这个属性，网络不能这样指定

In [ ]:
import torch
from torch import nn

# 我们可以在创建张量时指定存储设备
device = torch.device("cuda:0")
X = torch.ones(2, 3, device=device)
X.device

device(type='cuda', index=0)

In [5]:
import torch
from torch import nn

# 我们可以在创建张量时指定存储设备
# 可以直接写，['cpu', 'cuda', 'cuda:0', 'cuda:1']
X = torch.ones(2, 3, device='cpu')
X.device

device(type='cuda', index=0)

### set_default_tensor_type
张量, 模型都在GPU上

In [4]:
import torch
from torch import nn

torch.set_default_tensor_type('torch.cuda.FloatTensor')     # torch.cuda.DoubleTensor 双精度

# 张量
X = torch.ones(2, 3)
print(X, X.device)

net = nn.Sequential(nn.Linear(3, 1))
print(net[0].weight.data.device)

net(X)

tensor([[1., 1., 1.],
        [1., 1., 1.]]) cuda:0
cuda:0


tensor([[-0.8205],
        [-0.8205]], grad_fn=<AddmmBackward0>)

# 多GPU

- `multiprocessing`
  
  There are significant caveats to using CUDA models with `multiprocessing`; unless care is taken to meet the data handling requirements exactly, it is likely that your program will have incorrect or undefined behavior.

- `DataParallel`
  
  It is recommended to use `DistributedDataParallel`, instead of `DataParallel` to do multi-GPU training, **even if there is only a single node**.

## 单机多卡 single-node multi-GPU

### DataParallel

In [6]:
from torch import nn
net = nn.Sequential(nn.Linear(3, 1)).cuda()
net = nn.DataParallel(net)

先`net.cuda()` 还是先 `nn.DataParallel(net)`都行, 但是必须有`net.cuda()`

`net.to(device)`也行, 送到哪个GPU上无所谓, 反正都会再复制到所有指定的GPU上.

> to(device)版本

In [5]:
import torch
import os
from torch import nn
from torch.utils.data import DataLoader, Dataset

input_size = 5
output_size = 2

batch_size = 30
data_size = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class RandomDataset(Dataset):

    def __init__(self, size, length):
        self.len = length
        self.data = torch.randn(length, size)

    def __getitem__(self, index):
        return self.data[index]

    def __len__(self):
        return self.len

rand_loader = DataLoader(dataset=RandomDataset(input_size, data_size),
                         batch_size=batch_size, shuffle=True)

class Model(nn.Module):
    # Our model

    def __init__(self, input_size, output_size):
        super(Model, self).__init__()
        self.fc = nn.Linear(input_size, output_size)

    def forward(self, input):
        output = self.fc(input)
        print("\tIn Model: input size", input.size(),
              "output size", output.size())

        return output

model = Model(input_size, output_size).to(device)
if torch.cuda.device_count() > 1:
  print("Let's use", torch.cuda.device_count(), "GPUs!")
  # dim = 0 [30, xxx] -> [10, ...], [10, ...], [10, ...] on 3 GPUs
  model = nn.DataParallel(model)


for data in rand_loader:
    input = data.to(device)
    output = model(input)
    print("Outside: input size", input.size(),
          "output_size", output.size())

Let's use 2 GPUs!
	In Model: input size torch.Size([15, 5]) output size torch.Size([15, 2])
	In Model: input size torch.Size([15, 5]) output size torch.Size([15, 2])
Outside: input size torch.Size([30, 5]) output_size torch.Size([30, 2])
	In Model: input size torch.Size([15, 5]) output size torch.Size([15, 2])
	In Model: input size torch.Size([15, 5]) output size torch.Size([15, 2])
Outside: input size torch.Size([30, 5]) output_size torch.Size([30, 2])
	In Model: input size torch.Size([15, 5]) output size torch.Size([15, 2])
	In Model: input size torch.Size([15, 5]) output size torch.Size([15, 2])
Outside: input size torch.Size([30, 5]) output_size torch.Size([30, 2])
	In Model: input size torch.Size([5, 5]) output size torch.Size([5, 2])
	In Model: input size torch.Size([5, 5]) output size torch.Size([5, 2])
Outside: input size torch.Size([10, 5]) output_size torch.Size([10, 2])


> cuda()版本

In [1]:
import torch
import os
from torch import nn
from torch.utils.data import DataLoader, Dataset

input_size = 5
output_size = 2

batch_size = 30
data_size = 100

class RandomDataset(Dataset):

    def __init__(self, size, length):
        self.len = length
        self.data = torch.randn(length, size)

    def __getitem__(self, index):
        return self.data[index]

    def __len__(self):
        return self.len

rand_loader = DataLoader(dataset=RandomDataset(input_size, data_size),
                         batch_size=batch_size, shuffle=True)

class Model(nn.Module):
    # Our model

    def __init__(self, input_size, output_size):
        super(Model, self).__init__()
        self.fc = nn.Linear(input_size, output_size)

    def forward(self, input):
        output = self.fc(input)
        print("\tIn Model: input size", input.size(),
              "output size", output.size())

        return output

model = Model(input_size, output_size).cuda()
if torch.cuda.device_count() > 1:
  print("Let's use", torch.cuda.device_count(), "GPUs!")
  # dim = 0 [30, xxx] -> [10, ...], [10, ...], [10, ...] on 3 GPUs
  model = nn.DataParallel(model)


for data in rand_loader:
    input = data.cuda()
    output = model(input)
    print("Outside: input size", input.size(),
          "output_size", output.size())

Let's use 2 GPUs!
	In Model: input size torch.Size([15, 5]) output size torch.Size([15, 2])
	In Model: input size torch.Size([15, 5]) output size torch.Size([15, 2])
Outside: input size torch.Size([30, 5]) output_size torch.Size([30, 2])
	In Model: input size torch.Size([15, 5]) output size torch.Size([15, 2])
	In Model: input size torch.Size([15, 5]) output size torch.Size([15, 2])
Outside: input size torch.Size([30, 5]) output_size torch.Size([30, 2])
	In Model: input size torch.Size([15, 5]) output size torch.Size([15, 2])
	In Model: input size torch.Size([15, 5]) output size torch.Size([15, 2])
Outside: input size torch.Size([30, 5]) output_size torch.Size([30, 2])
	In Model: input size torch.Size([5, 5]) output size torch.Size([5, 2])
	In Model: input size torch.Size([5, 5]) output size torch.Size([5, 2])
Outside: input size torch.Size([10, 5]) output_size torch.Size([10, 2])


## DistributedDataParallel

- process group

  The devices to synchronize across are specified by the input process_group, **which is the entire world by default**.

  Creation of this class requires that torch.distributed to be already initialized, by calling `torch.distributed.init_process_group()`.

  This utility and multi-process distributed (single-node or multi-node) GPU training currently only achieves the best performance using the `NCCL` distributed backend. Thus NCCL backend is the recommended backend to use for GPU training.

- DistributedSampler
  
  DistributedDataParallel does not chunk or otherwise shard the input across participating GPUs; the user is responsible for defining how to do so, for example through the use of a `DistributedSampler`.

> launch

`torchrun` provides a superset of the functionality as `torch.distributed.launch` with the following additional functionalities:

- Worker failures are handled gracefully by restarting all workers.

- Worker `RANK` and `WORLD_SIZE` are assigned automatically.

- Number of nodes is allowed to change between minimum and maximum sizes (elasticity).

NOTE: `torchrun` is a python console script to the main module `torch.distributed.run` declared in the entry_points configuration in setup.py. It is equivalent to invoking `python -m torch.distributed.run`.

<https://pytorch.org/docs/stable/elastic/run.html#launcher-api>

```python
import torch
import torch.nn as nn

class ToyModel(nn.Module):
    def __init__(self):
        super(ToyModel, self).__init__()
        self.net1 = nn.Linear(10, 10)
        self.relu = nn.ReLU()
        self.net2 = nn.Linear(10, 5)

    def forward(self, x):
        return self.net2(self.relu(self.net1(x)))


if __name__ == "__main__":

    torch.distributed.init_process_group(backend="nccl")
    device = torch.distributed.get_rank()
    print(f"Start running basic DDP example on device {device}.")

    # create model and move it to GPU with id device
    model = ToyModel().to(device)
    ddp_model = nn.parallel.DistributedDataParallel(model, device_ids=[device])

    loss_fn = nn.MSELoss()
    optimizer = torch.optim.SGD(ddp_model.parameters(), lr=0.001)

    optimizer.zero_grad()
    outputs = ddp_model(torch.randn(20, 10))
    labels = torch.randn(20, 5).to(device)
    loss_fn(outputs, labels).backward()
    optimizer.step()
```

关键:
- `torch.distributed.init_process_group(backend="nccl")`
- `device = torch.distributed.get_rank()`
- `model = ToyModel().to(device)`, `ddp_model = nn.parallel.DistributedDataParallel(model, device_ids=[device])`

In [4]:
# --nproc_per_node=NUM_GPUS_YOU_HAVE
!torchrun --nproc_per_node=2 DDP-example.py

# or
!python -m torch.distributed.run --nproc_per_node=2 DDP-example.py

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
Start running basic DDP example on rank 1.
Start running basic DDP example on rank 0.
